# Submission 2: Reflection and the Lightweight Challenge: SOLUTION KEY
### ME 323 Module 1 (staff only)

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/I_beam_dimensions.jpg" alt="I-beam dimensions" width="220">

**The student notebook is now open-ended.** It hands out no lightweight
pipeline, no fill-in lines, and no checkpoint: students copy their own
Submission 1 code, choose their own confidence rule (lane, kernel, noise,
`z`, or a defended replacement), and argue for a design. This KEY therefore
does two jobs:

1. **Reference solution** — the class-default recipe worked end to end, so a
   grader knows what the default path produces (sections 1–3 below).
2. **Audit tools** — a sweep that maps where *defensible* designs land across
   every knob combination, and a `check_design` helper that reproduces any
   group's claimed numbers under their own stated knobs (section 2b).

**Suggested grading pass per group:** run their notebook top to bottom (no
checkpoints means every memo number must reproduce from their code — this is
the rubric's pass/fail code check); pull their stated knobs and final design
from the required end-of-section-2 printout; audit with `check_design`; place
them against the sweep table; then grade the memo prompts against the targets
at the bottom of this KEY.

## 0. Recall

Write before computing. Revise an answer when the later analysis changes your
understanding, and identify the evidence that motivated the revision.

1. Name the three modeled capacity branches and explain how the dominant-mode
   proxy is assigned. Which region of the (b, H_web) box does each own?
2. Pre-lab 1 calibrated σ_y, k, and τ_i. For each, one sentence: was the
   fitted value a correction to a handbook number, or the measurement of a
   property no handbook lists?
3. Distinguish epistemic, aleatory, and total predictive uncertainty. Which
   sigma drives explore-vs-exploit, and which belongs in a future-beam bound?
4. The equation query returned below its prediction; the GP query returned
   above its central prediction. Give one plausible reason for each miss.
5. Name the four model architectures from Submission 1 and the one-line idea of each.

### KEY: recall targets

Individually graded (7% of the module). Credit any defensible variant in the
student's own words; the targets are:

1. Bending (`P_bend`), flange-web separation (`P_sep`), lateral-torsional
   buckling (`P_LTB`); capacity is their minimum and the proxy label is the
   argmin — a modeled proxy, not an observed mechanism. Under the calibrated
   parameters, separation owns the thin-web edge (b ≲ 2 mm) at low-to-mid web
   heights, LTB takes over that same thin-web edge once the web is tall (thin
   flanges, roughly H_web ≳ 13 mm), and bending owns the broad remainder of
   the box.
2. σ_y (76 → 66.8 MPa) corrects a handbook number for printed material; k is
   a property of *this fixture* no handbook lists; τ_i started as a bulk-yield
   guess but the fitted 16.8 MPa measures a printed-interface property no
   handbook lists. A complete response takes a position rather than only reciting
   the numbers.
3. Epistemic = model ignorance, shrinks with data, drives explore-vs-exploit;
   aleatory = repeatability scatter, irreducible by more of the same tests;
   total = both combined, and only total belongs in a one-future-beam bound.
4. Equation beam (23% low): any named model-form candidate — mode-competition
   handoff mis-modeled near the aggressive design, calibration transferred
   from other geometries, print-to-print variability. GP beam (≈ on
   prediction): it interpolated near tested designs, so a near-zero miss is
   what an honest interpolation should produce — and says little about the
   model far from data.
5. A plain (pattern-match log str/w), B strength model (learn raw newtons, divide
   by mass after), C features (physics predictions as extra inputs),
   D residual error (physics first, GP learns its log error).

## 1. Your beam's test result

Students now paste their own Submission 1 setup and **write the interval
check themselves**; the cell below is the reference implementation of what
their code must do. Grade the logic, not the text: estimated vs measured mass
reported and separated, str/w on both denominators labeled,
σ_total = √(σ_epi² + 0.03²) around the *preregistered* central prediction,
and an explicit inside/outside verdict.

**Grader checks for section 1:**
- The interval uses the **model-basis** ratio (estimated-mass denominator)
  and the prediction actually filed with Submission 1 — not one recomputed
  after the result was known. Cross-check against the group's
  preregistration table.
- The trace-shape characterization (immediate / delayed / progressive /
  terminal loss) is evidence, not a mechanism label, and should be compared
  with the preregistered morphology.
- An outside-interval result must not be assigned a single cause; the memo
  target for prompt 1 below applies.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
TAU_I = 43.9e6                    # printed-interface shear strength (Pa) — starting
                                  # guess = bulk yield / sqrt(3)     (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    try:
        df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "no internet and no local copy -- download student_beams_B10_L150.csv "
            "from the course page into this notebook's folder and rerun") from e
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def estimated_mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_est_g"] = estimated_mass_g(df.b, df.H)
df["mass_delta_g"] = df.weight_g - df.mass_est_g
df["mass_delta_pct"] = 100*df.mass_delta_g/df.mass_est_g
print(len(df), "tested beams")

new = pd.DataFrame([
    dict(beam_id=16, b=1.10, H=13.25, strength_N=475.7,
         failure_note="equation-query result; observed morphology not supplied"),
    dict(beam_id=17, b=1.00, H=13.39, strength_N=445.8,
         failure_note="locked-GP-query result; observed morphology not supplied"),
])
new["weight_g"] = np.nan
new["mass_est_g"] = estimated_mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_est_g

# KEY reference implementation of the section-1 interval check students now
# write themselves. Grade their code against this logic, not this exact text.
b_mine, H_mine = None, None        # your Submission 1 design (mm)
P_mine = None                      # measured failure load (N)
mass_measured_mine = None          # measured printed-beam mass (g)
note_mine = ""                     # what the failure looked like
pred_median_sw = None              # copy model central prediction from Submission 1
pred_sigma_log = None              # copy epistemic sigma_log from Submission 1
if None not in (b_mine, H_mine, P_mine, mass_measured_mine,
                pred_median_sw, pred_sigma_log):
    mass_est_mine = estimated_mass_g(b_mine, H_mine)
    sw_mine_model_basis = P_mine / mass_est_mine
    sw_mine_measured_mass = P_mine / mass_measured_mine
    sigma_total_log = np.sqrt(pred_sigma_log**2 + 0.03**2)
    pred_lo_sw = pred_median_sw*np.exp(-2*sigma_total_log)
    pred_hi_sw = pred_median_sw*np.exp(2*sigma_total_log)
    inside_2sigma = pred_lo_sw <= sw_mine_model_basis <= pred_hi_sw
    print(f"your beam: ({b_mine}, {H_mine}), {P_mine} N")
    print("  observed failure note:", note_mine)
    print(f"  measured mass={mass_measured_mine:.2f} g; "
          f"estimated mass={mass_est_mine:.2f} g; "
          f"difference={mass_measured_mine-mass_est_mine:+.2f} g")
    print(f"  measured-mass str/w={sw_mine_measured_mass:.1f} N/g; "
          f"model-basis str/w={sw_mine_model_basis:.1f} N/g")
    print(f"  sigma_epi={pred_sigma_log:.3f}, sigma_total={sigma_total_log:.3f}")
    print(f"posterior-predictive interval: [{pred_lo_sw:.1f}, {pred_hi_sw:.1f}] N/g")
    print("inside recorded model +/-2 sigma interval:", inside_2sigma)
    print(f"class scoreboard: best tested so far {df.str_to_weight.max():.1f} N/g")

loaded from GitHub
15 tested beams


The model was trained on strength divided by estimated mass, so **the
interval check uses that same denominator**. Report the measured-mass ratio
too, but **never compare ratios with different denominators** as if they were
the same quantity. If the result lies outside the interval, distinguish
model-form error, print-to-print variability, and an unmodeled failure
mechanism. One test does not identify which.

## 2. The lightweight challenge — reference solution (class-default rule)

The student side states the class-default rule
($P_{lo} = e^{\mu_{\ln P} - z\,\sigma_{total}} \ge 700$ N with z = 2,
lane A, RBF, 3% noise) as a *starting point* and opens every piece of it to a
defended alternative. This cell is the default path worked end to end — the
anchor for groups that kept the default, and the baseline every alternative
is implicitly priced against.

In [2]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def Q_flange(p):
    return p["B"]*p["tf"]*(p["h"]/2 + p["tf"]/2)
def P_sep(p, tau_i):
    """Flange-web separation: shear flow vs strength along the printed layer lines."""
    return 2*tau_i*p["Ix"]*p["b"]/Q_flange(p)
def capacity(b, H, sy, k, tau_i):
    """Class model (2026-07-15): plain minimum of the three mode capacities."""
    p = section_props(b, H)
    return min(P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, tau_i):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k)
    return "separation" if Ps < min(Pb, Pl) else (
        "LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, TAU_CAL = 6.683e+07, 0.377, 1.676e+07
P_TARGET = 700.0

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern

# class-default model: lane A (log str/w), RBF, 3% noise, z = 2
X = df[["b", "H"]].values
fmu, fsd = X.mean(0), X.std(0) + 1e-12
y = np.log(df.str_to_weight.values)
ymean = y.mean()
gp = GaussianProcessRegressor(C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
                              alpha=0.03**2, normalize_y=False,
                              n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)

bg = np.linspace(1.0, 7.0, 63); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
Xg = np.column_stack([BB.ravel(), HH.ravel()])
mu_c, std = gp.predict((Xg - fmu)/fsd, return_std=True)
mass_grid = estimated_mass_g(Xg[:, 0], Xg[:, 1])
# strength = str/w * mass, so in logs: ln P = (mu + ymean) + ln(mass)
mu_lnP = mu_c + ymean + np.log(mass_grid)
sigma_aleatory = 0.03
sigma_total = np.sqrt(std**2 + sigma_aleatory**2)

P_lo = np.exp(mu_lnP - 2*sigma_total)
feasible = P_lo >= P_TARGET
median_strength = np.exp(mu_lnP)
masked = np.where(feasible, mass_grid, np.inf)
i = int(np.argmin(masked))
b_lt, H_lt = float(Xg[i, 0]), float(Xg[i, 1])
median_feasible = median_strength >= P_TARGET
i_median = int(np.argmin(np.where(median_feasible, mass_grid, np.inf)))
lighter_infeasible = (~feasible) & (mass_grid < mass_grid[i])
if lighter_infeasible.any():
    j = int(np.argmax(np.where(lighter_infeasible, mass_grid, -np.inf)))
else:
    j = None
print(f"REFERENCE LIGHTWEIGHT DESIGN (class-default rule): "
      f"b = {b_lt:.2f} mm, H_web = {H_lt:.2f} mm")
print(f"  mass {mass_grid[i]:.1f} g,  P_lo {P_lo[i]:.0f} N,  "
      f"posterior median {median_strength[i]:.0f} N")
print(f"  uncertainty allowance: posterior median - P_lo = "
      f"{median_strength[i]-P_lo[i]:.0f} N")
print(f"  median-only lightest design: b={Xg[i_median,0]:.2f}, "
      f"H_web={Xg[i_median,1]:.2f}, mass={mass_grid[i_median]:.1f} g, "
      f"median={median_strength[i_median]:.0f} N, P_lo={P_lo[i_median]:.0f} N")
print(f"  mass added by the 2-sigma rule versus median-only: "
      f"{mass_grid[i]-mass_grid[i_median]:.1f} g")
if j is not None:
    print(f"  closest-in-mass lighter infeasible grid point: b={Xg[j,0]:.2f}, "
          f"H_web={Xg[j,1]:.2f}, mass={mass_grid[j]:.3f} g "
          f"({mass_grid[i]-mass_grid[j]:.3f} g lighter), P_lo={P_lo[j]:.0f} N")
print(f"  calibrated-physics check: {capacity(b_lt, H_lt, SY_CAL, K_CAL, TAU_CAL):.0f} N, "
      f"mode {gov_mode(b_lt, H_lt, SY_CAL, K_CAL, TAU_CAL)}")
print("\nKEY note: the student notebook no longer prints a checkpoint. This")
print("reference answer (b = 5.16, H_web = 15.07, 21.9 g) is what the class-")
print("default recipe produces; groups that changed a knob should land elsewhere,")
print("and the sweep below says how far elsewhere is still ordinary.")

REFERENCE LIGHTWEIGHT DESIGN (class-default rule): b = 5.16 mm, H_web = 15.07 mm
  mass 21.9 g,  P_lo 701 N,  posterior median 778 N
  uncertainty allowance: posterior median - P_lo = 77 N
  median-only lightest design: b=4.39, H_web=15.07, mass=19.5 g, median=701 N, P_lo=640 N
  mass added by the 2-sigma rule versus median-only: 2.4 g
  closest-in-mass lighter infeasible grid point: b=2.45, H_web=9.66, mass=21.897 g (0.003 g lighter), P_lo=617 N
  calibrated-physics check: 689 N, mode bend

KEY note: the student notebook no longer prints a checkpoint. This
reference answer (b = 5.16, H_web = 15.07, 21.9 g) is what the class-
default recipe produces; groups that changed a knob should land elsewhere,
and the sweep below says how far elsewhere is still ordinary.


### Stress-test the assumed aleatory noise (reference for memo prompt 5)

Students choose which assumption to stress; the 1/3/10% noise sweep below is
the class-standard version most will run. A group that stressed a different
knob (kernel, model architecture, z) instead is fine **if the memo says why that knob is
the exposed one** — grade the substitution argument, not the deviation.

In [3]:
noise_design_rows = []
for pct in [1, 3, 10]:
    r = pct/100
    gp_r = GaussianProcessRegressor(
        C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
        alpha=r**2, normalize_y=False,
        n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)
    mu_r, epi_r = gp_r.predict((Xg-fmu)/fsd, return_std=True)
    total_r = np.sqrt(epi_r**2 + r**2)
    lo_r = np.exp(mu_r+ymean+np.log(mass_grid)-2*total_r)
    i_r = int(np.argmin(np.where(lo_r >= P_TARGET, mass_grid, np.inf)))
    noise_design_rows.append(dict(
        noise_pct=pct, b=Xg[i_r, 0], H_web=Xg[i_r, 1],
        mass_est_g=mass_grid[i_r],
        median_strength_N=np.exp(mu_r[i_r]+ymean)*mass_grid[i_r],
        lower_predictive_N=lo_r[i_r]))
noise_design_table = pd.DataFrame(noise_design_rows)
print(noise_design_table.round(2).to_string(index=False))

 noise_pct    b  H_web  mass_est_g  median_strength_N  lower_predictive_N
         1 4.10  13.20       20.87             726.24              703.24
         3 5.16  15.07       21.90             778.35              701.25
        10 5.74  13.02       25.48             860.89              700.17


## 2b. KEY-only: the territory of defensible answers, and an audit tool

Because there is no checkpoint, graders need two things: a map of where
reasonable designs land, and a way to reproduce any group's claimed numbers.

**The sweep.** Every lane × kernel × {1, 3, 10}% noise × z ∈ {1, 2, 3}
combination, each yielding its lightest feasible design. Read it as
territory, not truth:

- The **z = 2 row band** is the rubric's "presumptively reasonable
  territory." A design inside it needs only ordinary rationale.
- A design **lighter than anything in the sweep** almost certainly rests on a
  rule weaker than any combination here (z < 1, noise < 1%, or a
  median-only rule with padding bolted on). Check whether its own stated
  lower bound actually clears 700 N — if not, the rubric caps the challenge
  contribution at half marks.
- A design **heavier than the sweep's heavy edge** is padding unless the memo
  prices the grams (a larger z, a named model-form worry). Bare safety
  factors also cap at half marks.
- Lane B rows are worth a glance before grading a lane-B group: the strength
  target is the weakest lane on LOO, and its designs can sit far from the
  others. That is a memo conversation, not an automatic deduction.
- The **light edge of the sweep is lane D** (physics-residual): at low
  assumed noise it accepts thin-web designs near b ≈ 1.6 mm at ~19 g —
  deep in the separation/LTB proxy region, leaning on the physics between
  data points. Legitimate, but this is exactly where memo prompt 4 (the
  physics comparison) carries the weight: a model-D group that never discusses the
  mode there has not defended its light design.

**The audit.** `check_design(b, H, lane, kernel, noise_pct, z)` reproduces a
group's mass, median, lower bound, feasibility verdict, physics cross-check,
and padding under their own stated knobs. If their printed numbers do not
reproduce (beyond library wobble — roughly one grid step in the design, a few
N in the bounds), the code check fails before the rationale is graded. For a
group that replaced the rule entirely (different quantile, physics-based
constraint, etc.), reproduce their logic from their own code instead and
grade the argument; `check_design` still gives the nearest-default
comparison the memo should have priced itself against.

In [4]:
# Submission 1 GP toolkit (verbatim), so any lane/kernel a group chose can be rebuilt here.
def make_kernel(kernel, ndim):
    if kernel == "RBF":
        return C(1.0, (1e-3, 1e3)) * RBF([1.0]*ndim, (1e-1, 30.0))
    elif kernel == "Matern":
        return C(1.0, (1e-3, 1e3)) * Matern([1.0]*ndim, (1e-1, 30.0), nu=2.5)
    raise ValueError(f"unknown kernel {kernel!r}: use 'RBF' or 'Matern'")

def build_feats(bq, Hq, feats):
    bq, Hq = np.atleast_1d(np.asarray(bq, float)), np.atleast_1d(np.asarray(Hq, float))
    cols = {"b": bq, "H": Hq}
    if "logP" in feats or "stab" in feats:
        pp = [section_props(b, H) for b, H in zip(bq, Hq)]
        Pb = np.array([P_bend(p, SY_CAL) for p in pp])
        Pl = np.array([P_LTB(p, SY_CAL, K_CAL) for p in pp])
        Ps = np.array([P_sep(p, TAU_CAL) for p in pp])
        cols["logP"] = np.log(np.minimum(Pb, np.minimum(Ps, Pl)))
        cols["stab"] = Pl/Pb
    return np.column_stack([cols[f] for f in feats])

def fit_gp(data, alpha=0.03**2, feats=("b", "H"), target="log_sw", kernel="RBF"):
    Xf = data[list(feats)].values.astype(float)
    fmu_, fsd_ = Xf.mean(0), Xf.std(0) + 1e-12
    if target == "log_sw":
        yf = np.log(data.strength_N.values /
                    estimated_mass_g(data.b.values, data.H.values))
    elif target == "log_strength":
        yf = np.log(data.strength_N.values.astype(float))
    else:
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, TAU_CAL)
                          for b, H in zip(data.b, data.H)])
        yf = np.log(data.strength_N.values) - np.log(Pphys)
    ym_ = yf.mean()
    gp_ = GaussianProcessRegressor(make_kernel(kernel, Xf.shape[1]), alpha=alpha,
                                   normalize_y=False,
                                   n_restarts_optimizer=5, random_state=0)
    gp_.fit((Xf - fmu_) / fsd_, yf - ym_)
    return gp_, fmu_, fsd_, ym_

def predict_sw(gp_, fmu_, fsd_, ym_, bq, Hq, target, feats):
    Xq = build_feats(bq, Hq, feats)
    mu, sd = gp_.predict((Xq - fmu_)/fsd_, return_std=True)
    mu = mu + ym_
    mass = estimated_mass_g(np.asarray(bq, float), np.asarray(Hq, float))
    if target == "log_sw":
        sw = np.exp(mu)
    elif target == "log_strength":
        sw = np.exp(mu)/mass
    else:
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, TAU_CAL)
                          for b, H in zip(np.atleast_1d(bq), np.atleast_1d(Hq))])
        sw = Pphys*np.exp(mu)/mass
    return sw, sd

LANES = {
    "A plain":    dict(feats=("b", "H"), target="log_sw"),
    "B strength model": dict(feats=("b", "H"), target="log_strength"),
    "C features": dict(feats=("b", "H", "logP", "stab"), target="log_sw"),
    "D residual error": dict(feats=("b", "H"), target="log_residual"),
}

def fit_lane(data, lane, alpha=0.03**2, kernel="RBF"):
    cfg = LANES[lane]
    d2 = data.copy()
    Xf = build_feats(d2.b.values, d2.H.values, cfg["feats"])
    for jj, f in enumerate(cfg["feats"]):
        d2[f] = Xf[:, jj]
    gp_, fmu_, fsd_, ym_ = fit_gp(d2, alpha=alpha, feats=cfg["feats"],
                                  target=cfg["target"], kernel=kernel)
    return gp_, fmu_, fsd_, ym_, cfg

# In every lane the deterministic pieces (mass, physics capacity) drop out of the
# variance, so the GP's sd IS the sigma of ln(strength). One helper serves all four.
def lane_grid_bounds(lane, kernel, noise_pct):
    """Posterior median strength and sigma_lnP over the standard grid."""
    r = noise_pct/100
    gp_, fmu_, fsd_, ym_, cfg = fit_lane(df, lane, alpha=r**2, kernel=kernel)
    sw, sd = predict_sw(gp_, fmu_, fsd_, ym_, Xg[:, 0], Xg[:, 1],
                        cfg["target"], cfg["feats"])
    med_N = sw * mass_grid
    tot = np.sqrt(sd**2 + r**2)
    return med_N, tot

sweep_rows = []
for lane in LANES:
    for kernel in ("RBF", "Matern"):
        for noise_pct in (1, 3, 10):
            med_N, tot = lane_grid_bounds(lane, kernel, noise_pct)
            for z in (1.0, 2.0, 3.0):
                lo = med_N * np.exp(-z*tot)
                ok = lo >= P_TARGET
                if not ok.any():
                    sweep_rows.append(dict(lane=lane, kernel=kernel,
                                           noise_pct=noise_pct, z=z, b=np.nan,
                                           H_web=np.nan, mass_g=np.nan,
                                           median_N=np.nan, P_lo=np.nan))
                    continue
                k = int(np.argmin(np.where(ok, mass_grid, np.inf)))
                sweep_rows.append(dict(lane=lane, kernel=kernel,
                                       noise_pct=noise_pct, z=z,
                                       b=Xg[k, 0], H_web=Xg[k, 1],
                                       mass_g=mass_grid[k],
                                       median_N=med_N[k], P_lo=lo[k]))
sweep = pd.DataFrame(sweep_rows)
print(sweep.round(2).to_string(index=False))

z2 = sweep[(sweep.z == 2.0) & sweep.mass_g.notna()]
print(f"\nz = 2 territory across all lanes/kernels/noise: "
      f"mass {z2.mass_g.min():.1f}-{z2.mass_g.max():.1f} g, "
      f"b {z2.b.min():.2f}-{z2.b.max():.2f} mm, "
      f"H_web {z2.H_web.min():.2f}-{z2.H_web.max():.2f} mm")
allz = sweep[sweep.mass_g.notna()]
print(f"all z in (1, 2, 3):                            "
      f"mass {allz.mass_g.min():.1f}-{allz.mass_g.max():.1f} g")

            lane kernel  noise_pct   z    b  H_web  mass_g  median_N   P_lo
         A plain    RBF          1 1.0 4.48  14.88   20.02    727.54 701.03
         A plain    RBF          1 2.0 4.10  13.20   20.87    726.24 703.24
         A plain    RBF          1 3.0 4.29  13.39   21.18    736.19 701.38
         A plain    RBF          3 1.0 4.68  14.88   20.61    737.31 701.75
         A plain    RBF          3 2.0 5.16  15.07   21.90    778.35 701.25
         A plain    RBF          3 3.0 4.97  13.20   23.22    792.00 702.26
         A plain    RBF         10 1.0 4.87  13.20   22.96    776.61 700.42
         A plain    RBF         10 2.0 5.74  13.02   25.48    860.89 700.17
         A plain    RBF         10 3.0 6.81  13.02   28.31    956.62 701.65
         A plain Matern          1 1.0 4.58  15.07   20.11    729.57 704.69
         A plain Matern          1 2.0 4.19  13.39   20.91    728.33 701.79
         A plain Matern          1 3.0 4.39  13.39   21.44    743.51 701.33
         A p

In [5]:
def check_design(b, H, lane="A plain", kernel="RBF", noise_pct=3, z=2.0):
    """Audit one group's claimed design under their own stated knobs.

    Prints everything the rubric needs: mass, median, lower bound,
    feasibility under their rule, the physics cross-check, and the padding
    relative to the lightest design their own rule accepts."""
    r = noise_pct/100
    gp_, fmu_, fsd_, ym_, cfg = fit_lane(df, lane, alpha=r**2, kernel=kernel)
    sw, sd = predict_sw(gp_, fmu_, fsd_, ym_, [b], [H],
                        cfg["target"], cfg["feats"])
    mass = float(estimated_mass_g(b, H))
    med = float(sw[0]) * mass
    tot = float(np.sqrt(sd[0]**2 + r**2))
    lo = med * np.exp(-z*tot)
    med_N, tot_g = lane_grid_bounds(lane, kernel, noise_pct)
    lo_g = med_N * np.exp(-z*tot_g)
    ok = lo_g >= P_TARGET
    print(f"design ({b}, {H}) under lane={lane!r}, kernel={kernel}, "
          f"noise={noise_pct}%, z={z}:")
    print(f"  estimated mass {mass:.1f} g | median {med:.0f} N | "
          f"sigma_total {tot:.3f} | P_lo {lo:.0f} N")
    print(f"  clears 700 N under this rule: {lo >= P_TARGET}")
    print(f"  physics cross-check: capacity "
          f"{capacity(b, H, SY_CAL, K_CAL, TAU_CAL):.0f} N, "
          f"mode {gov_mode(b, H, SY_CAL, K_CAL, TAU_CAL)}")
    if ok.any():
        k = int(np.argmin(np.where(ok, mass_grid, np.inf)))
        print(f"  lightest design THEIR rule accepts: ({Xg[k,0]:.2f}, {Xg[k,1]:.2f}), "
              f"{mass_grid[k]:.1f} g -> padding carried: {mass - mass_grid[k]:+.1f} g")
    else:
        print("  no grid design clears 700 N under this rule")

# example: the reference design audited under the class-default knobs
check_design(5.16, 15.07)

design (5.16, 15.07) under lane='A plain', kernel=RBF, noise=3%, z=2.0:
  estimated mass 21.9 g | median 778 N | sigma_total 0.052 | P_lo 701 N
  clears 700 N under this rule: True
  physics cross-check: capacity 689 N, mode bend
  lightest design THEIR rule accepts: (5.16, 15.07), 21.9 g -> padding carried: -0.0 g


### KEY-only: synthetic example result for the refit demo

The next cell fills the section-1 blanks with an invented but plausible result
(the Submission 1 default-recipe design failing 3% below its predicted median
strength) purely so the refit section below shows real output in this KEY.
It is labeled synthetic and is NOT a test result. Students use their own beam.

In [6]:
b_mine, H_mine = 1.39, 14.88
mu_demo = gp.predict((np.array([[b_mine, H_mine]])-fmu)/fsd)[0] + ymean
P_mine = round(0.97*float(np.exp(mu_demo))*estimated_mass_g(b_mine, H_mine), 1)
mass_measured_mine = round(float(estimated_mass_g(b_mine, H_mine)), 2)
note_mine = "synthetic KEY example -- not a real test"
print(f"KEY demo beam: ({b_mine}, {H_mine}), {P_mine} N (synthetic)")

KEY demo beam: (1.39, 14.88), 369.4 N (synthetic)


## 3. What your beam changes: refit with your own result

Students append their beam as row 18, refit with **their own Submission 1
knobs**, and print the 700 N design and best-str/w pick before and after. The
reference implementation below uses the class default.

**Grader checks for section 3:** the refit knobs match the knobs the group
actually used in section 2 (a silent switch here is a red flag); movement
claims are quantified in grams and millimeters; "nothing moved" is accepted
as a finding when connected to the noise and length-scale assumptions.

In [7]:
if None in (b_mine, H_mine, P_mine):
    print("Enter your beam in section 1 first; this cell is skipped without it.")
else:
    mine = pd.DataFrame([dict(beam_id=18, b=b_mine, H=H_mine, strength_N=P_mine,
                              weight_g=mass_measured_mine,
                              mass_est_g=estimated_mass_g(b_mine, H_mine),
                              failure_note=note_mine)])
    df18 = pd.concat([df, mine], ignore_index=True)
    df18["str_to_weight"] = df18.strength_N / df18.mass_est_g
    X17 = df18[["b", "H"]].values
    fmu17, fsd17 = X17.mean(0), X17.std(0) + 1e-12
    y17 = np.log(df18.str_to_weight.values); ym17 = y17.mean()
    gp17 = GaussianProcessRegressor(
        C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
        alpha=0.03**2, normalize_y=False,
        n_restarts_optimizer=5, random_state=0).fit((X17-fmu17)/fsd17, y17-ym17)
    mu17, epi17 = gp17.predict((Xg-fmu17)/fsd17, return_std=True)
    tot17 = np.sqrt(epi17**2 + sigma_aleatory**2)
    lo17 = np.exp(mu17 + ym17 + np.log(mass_grid) - 2*tot17)
    i17 = int(np.argmin(np.where(lo17 >= P_TARGET, mass_grid, np.inf)))
    i_sw16 = int(np.argmax(mu_c))
    i_sw17 = int(np.argmax(mu17))
    mu_at_mine, _ = gp.predict((np.array([[b_mine, H_mine]])-fmu)/fsd,
                               return_std=True)
    pred_N_at_mine = np.exp(mu_at_mine[0]+ymean)*estimated_mass_g(b_mine, H_mine)
    print("lightweight design, 17 beams -> 18 beams:")
    print(f"  before: b={b_lt:.2f}, H_web={H_lt:.2f}, mass={mass_grid[i]:.1f} g")
    print(f"  after:  b={Xg[i17,0]:.2f}, H_web={Xg[i17,1]:.2f}, "
          f"mass={mass_grid[i17]:.1f} g")
    print("best posterior-median str/w point, 17 -> 18 beams:")
    print(f"  before: ({Xg[i_sw16,0]:.2f}, {Xg[i_sw16,1]:.2f});  "
          f"after: ({Xg[i_sw17,0]:.2f}, {Xg[i_sw17,1]:.2f})")
    print(f"  17-beam model median at your design: {pred_N_at_mine:.0f} N; "
          f"your measured strength: {P_mine:.0f} N")
    print("If nothing moved: one test rarely re-shapes a 17-beam posterior far")
    print("from the tested point. Whether it SHOULD have moved more is a memo")
    print("question about the noise and length-scale assumptions, not a code bug.")

lightweight design, 17 beams -> 18 beams:
  before: b=5.16, H_web=15.07, mass=21.9 g
  after:  b=5.16, H_web=15.07, mass=21.9 g
best posterior-median str/w point, 17 -> 18 beams:
  before: (1.00, 13.20);  after: (1.00, 13.20)
  17-beam model median at your design: 381 N; your measured strength: 369 N
If nothing moved: one test rarely re-shapes a 17-beam posterior far
from the tested point. Whether it SHOULD have moved more is a memo
question about the noise and length-scale assumptions, not a code bug.


## Memo

Write it in markdown cells below this one, answering the prompts in order —
400 to 800 words. This memo is the module's primary assessed artifact, and
this submission generates no numbers for you: every figure you cite must come
out of code your group ran above.

0. Card row 11: before the prompts, complete the **Update** row of your
   decision card — what did the result change: your model, your confidence,
   or your next design? One assumption, named.
1. Reflection: report measured and estimated mass, use the model-basis ratio for
   the interval check, and compare the observed strength and failure note with
   what your group **preregistered** — the section-1 interval built from your
   preregistered prediction and sigma, the expected morphology, and the named
   most-likely-to-break assumption. Was the surprise (or its absence) the one
   you priced?
2. Margin: state, in newtons and grams, what your confidence rule bought.
   That takes three designs from your own analysis: your final pick, the
   lightest design a median-only rule would accept, and a lighter design
   your rule rejects. If you cannot produce the second and third, your rule
   was never really tested.
3. Confidence rule: defend your `z` (or whatever replaced it). Under the
   independent Gaussian log-noise model, the chance that one future measured
   beam falls below a two-sigma lower predictive bound is 2.28%. Kernel and
   model-form errors are outside that probability statement.
4. Physics comparison: cite the calibrated capacity and dominant-mode proxy at your
   design. If the physics disagrees with the GP constraint, explain which
   evidence you prioritize.
5. Sensitivity: which assumption did you stress-test, what moved, and is your
   design decision assumption-sensitive? If you departed from the 1/3/10%
   noise sweep, defend the substitution.
6. One more test: provide coordinates and say whether posterior median,
   epistemic sigma, or proximity to the feasibility boundary motivates it.
7. Redesign: from section 3, did your result move the lightweight design or
   the best-str/w pick? Whichever way it went, defend what you would print
   next, and connect the movement (or its absence) to the noise and
   length-scale assumptions.

**Individual postscript (each student, 3–5 sentences, after the group memo):**
one place you agreed with the group's decision and one place you would have
decided differently, with the evidence you'd cite. Both agreement and
disagreement should be tied to evidence.

## KEY: memo targets

Because students choose their own rule, most prompts no longer have a single
right number. For each, the target is a *shape of argument*; the sweep and
`check_design` supply the numbers to hold it against.

0. **Card row 11.** One named assumption, one named change (model /
   confidence / next design). "We learned a lot" with nothing named earns no
   credit.
1. **Reflection.** Like denominators (model-basis ratio vs the GP trained on
   it); the comparison is against the section-1 interval built from the
   *preregistered* central prediction and sigma, and the *preregistered*
   morphology, quoted, not paraphrased from memory. Outside-interval results
   must hold model-form error, print variability, and unmodeled mechanism
   apart as live candidates. Strong: "the miss was (not) the one we priced,
   because…". Weak: post-hoc single-cause stories, or "agreed well" with no
   numbers.
2. **Margin.** Three designs, all from their own analysis: final pick,
   median-only pick, and a lighter rejected design — stated in N and g.
   Verify all three with `check_design` under their knobs. A memo that cannot
   produce the rejected design never exercised its own rule; that caps this
   prompt regardless of prose quality.
3. **Confidence rule.** z = 2 defended via the 2.28% one-sided tail *with its
   scope stated* (one future beam, independent Gaussian log-noise, kernel and
   model-form outside it) is full credit. Another z, or a replacement rule,
   is graded on whether its consequence is priced in grams and its
   probability claim is scoped as carefully. An unscoped "95% safe" is the
   canonical weak answer.
4. **Physics comparison.** The calibrated capacity and proxy mode at *their*
   design, from their code (`check_design` prints both). Agreement between
   GP and physics is supporting evidence, not independent validation — both
   were informed by the same 17-beam campaign. Disagreement demands a stated
   priority and a reason.
5. **Sensitivity.** What moved, in mm and g, and the verdict
   "assumption-sensitive or not." Stability under their sweep is local
   robustness, not proof the assumption is right — strong memos say so. A
   substituted stress-test needs a one-sentence exposure argument.
6. **One more test.** Coordinates plus a named motive (median, epistemic
   sigma, or feasibility boundary). Near the active lower-bound contour —
   especially where the proxy mode changes or sigma is large — is the strong
   region; a point far from both the boundary and any uncertainty is
   decoration.
7. **Redesign.** Either outcome (moved / did not move) is defensible; credit
   quantified movement tied to the noise and length-scale assumptions.
   Automatic "refit found a new optimum, we would print it" with no
   connection to the observation is the canonical weak answer.

**Half-marks caps (from the rubric, applied to the challenge):** a design
whose own stated lower bound sits under 700 N; or padding beyond the group's
own lightest confident design defended only by a bare safety factor. The
sweep's z = 2 band marks presumptively reasonable territory on the heavy
side.